In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append('..')
from modules.source import OUTPUT_FOLDER_PATH, parquet_daily, parquet_monthly, Sources

df_monthly = pd.read_parquet(parquet_monthly)
df_daily = pd.read_parquet(parquet_daily)

plt.rcParams.update({ 
   "figure.dpi": 150, "figure.titlesize": 14, "figure.titleweight": "bold", "savefig.bbox": "tight",
   "axes.titlesize": 9, "axes.titleweight": "bold", "axes.labelweight": "bold",
   "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.4,
   "grid.color": "#9ca3af", "grid.linewidth": 0.4,
   "lines.color": "#374151", "lines.linewidth": 0.8,
})

DISPLAY_NAMES = {
   Sources.rate_differential: "FFR-OPR Spread",
   Sources.crude_palm_oil: "Crude Palm Oil",
   Sources.brent_oil: "Brent Oil",
   Sources.UST_10Y: "UST 10Y Yield",
   Sources.cpi_inflation: " CPI Index",
   Sources.IPI: "IPI Index",
   Sources.HY_OAS: "HY OAS",

   Sources.financials: "Financials", 
   Sources.technology: "Technology", 
   Sources.energy: "Energy", 
   Sources.REITs: "REITs", 
   Sources.industrial_products: "Industrial Products", 
   Sources.plantation: "Plantation"
}

In [2]:
def plot(vars: list[str], title: str) -> None:
   fig, axes = plt.subplots(len(vars), 2, figsize=(10, 2 * len(vars)), sharex=True, layout="constrained")

   axes[0, 0].set_title("Level")
   axes[0, 1].set_title(r"Change ($\Delta$ / Log Return)")
   axes[0, 1].scatter([], [], color="#dc2626", label="outliers")
   axes[0, 1].legend(fontsize=6, loc="upper left", frameon=False)

   for (ax_lvl, ax_chg), v in zip(axes, vars):
      level, change = df_monthly[(v, "price")], df_monthly[(v, "change")]
      ax_lvl.plot(level)
      ax_lvl.set_ylabel(DISPLAY_NAMES.get(v, v))
      ax_chg.plot(change, color="#061b93")
      ax_chg.axhline(0)

      abs_z = ((change - change.mean()) / change.std()).abs()
      outlier_date = abs_z[abs_z > 2.5].nlargest(3).index
      ax_chg.scatter(outlier_date, change.loc[outlier_date], zorder=3, color="#dc2626", s=15)

   fig.suptitle(title)
   fig.savefig(f"{OUTPUT_FOLDER_PATH}/{title.lower().replace(" ", "_")}.png")
   plt.close()

independant_vars = [Sources.crude_palm_oil, Sources.brent_oil, Sources.cpi_inflation, Sources.IPI, Sources.rate_differential, Sources.UST_10Y, Sources.VIX, Sources.USDMYR, Sources.HY_OAS]
dependant_vars = [Sources.KLCI, Sources.financials, Sources.technology, Sources.energy, Sources.REITs, Sources.industrial_products, Sources.plantation]
plot(independant_vars, "Variance Question Independant Variables Charts")
plot(dependant_vars, "Variance Question Dependant Variables Charts")